# Scanning Optimizations

## Partition Pruning

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Turn off adaptive query optimization for this notebook
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [0]:
# Get AQE status
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
df = spark.read.format('csv')\
    .option('header', 'true')\
    .option('inferSchema', 'true')\
    .load('/Volumes/dbacademy/labuser14586003_1778589031/data/BigMart Sales.csv')


In [0]:
display(df)

Spark creates logical partitions on top of our data. But how does spark decide how many partitions to create? 
This is usually determined based on the input data size. The default partition size is 128MB but this can be manipulated using `spark.conf.set("spark.sql.files.maxPartitionBytes")`

It's recommended to use between 100 - 200MB

In [0]:
# Check the default partition size
spark.conf.get("spark.sql.files.maxPartitionBytes")

In [0]:
# Check the no. of partitions
df.rdd.getNumPartitions()

We have one partition because the data size is less than 128M. So, spark just read all the data into one logica partition. But we can force spark to read the data into a number of partitions of our choice using `spark.conf.set("spark.sql.files.maxPartitionBytes", our_value)`

In [0]:
# Change the default partition size to 128KB
spark.conf.set("spark.sql.files.maxPartitionBytes", 131072)

In [0]:
# Let's read the file again
df = spark.read.format('csv')\
    .option('header','true')\
    .option('inferSchema','true')\
    .load('/Volumes/dbacademy/labuser14586003_1778589031/data/BigMart Sales.csv')


In [0]:
# Check the no. of partitions again
df.rdd.getNumPartitions()

Wow! spark has just read the file into 7 partitions. We can check which records belong to which partion usi spark_partition_id

In [0]:
df.withColumn("partition_id", spark_partition_id())\
    .groupBy("partition_id")\
    .agg(count("*").alias("count"))\
    .orderBy("partition_id")\
    .display()

Notice how we have data on only two partitions. The other partitions are empty

In [0]:
# Change back to the default partition size
spark.conf.set("spark.sql.files.maxPartitionBytes", 128 * 1024 * 1024)
spark.conf.get("spark.sql.files.maxPartitionBytes")

We can repartition to force data on every partition

In [0]:
# Let's read the file again
df = spark.read.format('csv')\
    .option('header','true')\
    .option('inferSchema','true')\
    .load('/Volumes/dbacademy/labuser14586003_1778589031/data/BigMart Sales.csv')


In [0]:
df.rdd.getNumPartitions()

In [0]:
# Repartition to 7 partitions
df = df.repartition(7)

In [0]:
df.rdd.getNumPartitions()

In [0]:
# Get the count of rows in each partition
df.withColumn("partition_id", spark_partition_id())\
    .groupBy("partition_id")\
    .agg(count("*").alias("count"))\
    .orderBy("partition_id")\
    .display()

**Wow! Now we have records for the 7 partitions**